# 🎬 Recomendador de Filmes
## Matemática para Inteligência Artificial

**Temas:** vetores, matrizes, transformações lineares, SVD, derivadas,
integrais, gradiente descendente e métodos numéricos.

Este projeto explora conceitos matemáticos fundamentais utilizados em
Inteligência Artificial por meio da construção e análise de um sistema
de recomendação de filmes.

In [1]:
# ============================================
# CONFIGURAÇÃO DO PROJETO
# ============================================

AUTOR = "Ronildo Batista do Nascimento"

import numpy as np
import matplotlib.pyplot as plt

# Semente fixa para garantir resultados reproduzíveis
SEMENTE = 123456
np.random.seed(SEMENTE)

print(f"Projeto: Recomendador de Filmes")
print(f"Autor: {AUTOR}")
print(f"Semente utilizada: {SEMENTE}")

Projeto: Recomendador de Filmes
Autor: Ronildo Batista do Nascimento
Semente utilizada: 123456


---
# Parte 0 — Fundamentos com NumPy 🔥

Nesta etapa são demonstradas operações fundamentais com vetores utilizando NumPy, incluindo soma, multiplicação elemento a elemento, produto interno e cálculo da norma.

In [2]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("Soma:            ", a + b)
print("Multiplicação:   ", a * b)          # elemento a elemento
print("Produto interno: ", np.dot(a, b))   # 1*4 + 2*5 + 3*6 = 32
print("Norma (tamanho): ", np.linalg.norm(a))

Soma:             [5 7 9]
Multiplicação:    [ 4 10 18]
Produto interno:  32
Norma (tamanho):  3.7416573867739413


Produto Interno entre Vetores
Como demonstração, é calculado o produto interno entre os vetores a e c. Essa operação será posteriormente importante para medir a similaridade entre vetores no sistema de recomendação.

In [3]:
# Vetor utilizado na demonstração
c = np.array([2, 0, 1])

# Produto interno entre os vetores a e c
produto = np.dot(a, c)

print("Produto interno a·c =", produto)

Produto interno a·c = 5


---
# Parte 1 — Vetores e Similaridade: quem gosta do mesmo que você? 🤝

Nosso recomendador começa com uma **matriz de notas**: cada **linha é um usuário**, cada **coluna é um filme**, e cada valor é a nota (0 a 5) que o usuário deu. Nota 0 = não assistiu.

| | Matrix | Titanic | Toy Story | Vingadores | Barbie |
|---|---|---|---|---|---|
| **Ana** | 5 | 1 | 2 | 5 | 1 |
| **Bruno** | 4 | 0 | 1 | 5 | 2 |
| **Carla** | 1 | 5 | 4 | 1 | 5 |
| **Davi** | 0 | 4 | 5 | 2 | 4 |
| **Você** | 5 | 2 | 1 | **0** | 1 |

Repare: **você ainda não assistiu Vingadores**. O objetivo do recomendador é decidir se vale a pena.

A ideia central: **cada usuário é um vetor** no espaço dos filmes. Usuários com "gostos parecidos" são vetores que apontam para a mesma direção. Medimos isso com a **similaridade por cosseno**:

$$\text{sim}(\vec{u}, \vec{v}) = \cos(\theta) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \, \|\vec{v}\|}$$

- Resultado **perto de 1** → gostos muito parecidos
- Resultado **perto de 0** → gostos sem relação

In [ ]:
filmes = ["Matrix", "Titanic", "Toy Story", "Vingadores", "Barbie"]
usuarios = ["Ana", "Bruno", "Carla", "Davi", "Você"]

R = np.array([
    [5, 1, 2, 5, 1],   # Ana
    [4, 0, 1, 5, 2],   # Bruno
    [1, 5, 4, 1, 5],   # Carla
    [0, 4, 5, 2, 4],   # Davi
    [5, 2, 1, 0, 1],   # Você  (0 em Vingadores = não assistiu)
], dtype=float)

print("Formato da matriz (linhas, colunas):", R.shape)
print("Vetor do usuário 'Você':", R[4])

**Exercício 1.1** — Complete a função de similaridade por cosseno usando `np.dot` e `np.linalg.norm` (é a fórmula acima, traduzida para Python).

In [ ]:
def similaridade_cosseno(u, v):
    # 👉 SEU CÓDIGO AQUI: complete o numerador e o denominador
    numerador   = ...          # produto interno entre u e v
    denominador = ...          # norma de u vezes norma de v
    return numerador / denominador

# Teste: similaridade de você com cada usuário
voce = R[4]
for i, nome in enumerate(usuarios[:4]):
    s = similaridade_cosseno(voce, R[i])
    print(f"Similaridade Você × {nome}: {s:.3f}")

**❓ Pergunta 1.1** — Qual usuário é mais parecido com "Você"? Isso faz sentido olhando a tabela de notas? Que nota esse usuário deu para **Vingadores** — e, com base nisso, o sistema deveria ou não recomendar o filme?

**✍️ Sua resposta:**

*(escreva aqui)*

**❓ Pergunta 1.2 (crítica ao modelo)** — Combinamos que "nota 0 = não assistiu", mas o cosseno trata esse 0 como se fosse a **nota mínima**, ou seja, como se você tivesse odiado o filme. Que efeito isso tem sobre as similaridades calculadas? Sugira uma forma de tratar os filmes não assistidos que não confunda "não vi" com "detestei".

**✍️ Sua resposta:**

*(escreva aqui)*

---
# Parte 2 — Matrizes como Transformações: mexendo no espaço 🌀

Uma matriz não é só uma "tabela de números": quando multiplicamos uma matriz por um vetor, ela **transforma** o vetor — gira, estica, espelha. É assim que redes neurais processam dados: camada após camada de transformações lineares.

A matriz de **rotação** por um ângulo $\theta$ é:

$$M_{rot} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Vamos ver isso acontecendo com uma nuvem de pontos em forma de seta:

In [ ]:
# Uma "seta" feita de pontos 2D (cada coluna é um ponto)
seta = np.array([
    [0, 1, 2, 3, 3, 4, 3, 3, 2, 1, 0],
    [0.5, 0.5, 0.5, 0.5, 1, 0, -1, -0.5, -0.5, -0.5, -0.5]
])

def matriz_rotacao(graus):
    t = np.radians(graus)
    return np.array([[np.cos(t), -np.sin(t)],
                     [np.sin(t),  np.cos(t)]])

# 👉 SEU CÓDIGO AQUI: crie a matriz de rotação de 45 graus
#    e aplique na seta usando o operador @ (multiplicação de matrizes)
M = ...
seta_girada = ...   # dica: M @ seta

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(seta[0], seta[1], 'o-', label="Original")
ax.plot(seta_girada[0], seta_girada[1], 'o-', label="Girada 45°")
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_aspect('equal'); ax.legend(); ax.set_title("Transformação linear: rotação")
plt.show()

**Exercício 2.1** — Agora aplique na seta **original** a matriz abaixo e plote o resultado:

$$M_2 = \begin{bmatrix} 2 & 0 \\ 0 & 0.5 \end{bmatrix}$$

In [ ]:
# 👉 SEU CÓDIGO AQUI (3-4 linhas): defina M2, transforme a seta e plote
M2 = ...
seta_2 = ...

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(seta[0], seta[1], 'o-', label="Original")
ax.plot(seta_2[0], seta_2[1], 'o-', label="Transformada por M2")
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_aspect('equal'); ax.legend(); ax.set_title("O que M2 faz?")
plt.show()

**❓ Pergunta 2.1** — Descreva com suas palavras o que a matriz $M_2$ fez com a seta (o que aconteceu no eixo x? e no eixo y?).

**✍️ Sua resposta:**

*(escreva aqui)*

---
# Parte 3 — SVD: encontrando os "gostos escondidos" 🔍

Agora a estrela do trabalho. A **Decomposição em Valores Singulares (SVD)** quebra qualquer matriz em três:

$$R = U \, \Sigma \, V^T$$

Intuição no nosso recomendador: a SVD descobre **"gostos latentes"** — por exemplo, "gosta de ação" ou "gosta de drama" — sem ninguém ter dito isso a ela. Os **valores singulares** (em $\Sigma$) dizem a **importância** de cada gosto latente.

Vamos usar uma matriz maior de notas, gerada com a sua matrícula (por isso seus números serão diferentes dos do colega!):

In [ ]:
# Matriz de notas 20 usuários × 8 filmes, gerada a partir de 2 "gostos" escondidos:
#   gosto 0 = AÇÃO   |   gosto 1 = DRAMA
n_usuarios, n_filmes = 20, 8

gosto_usuarios = np.random.rand(n_usuarios, 2)   # quanto cada usuário tem de cada gosto
gosto_usuarios[:10, 1] *= 0.15                   # usuários 0-9:  fãs de AÇÃO
gosto_usuarios[10:, 0] *= 0.15                   # usuários 10-19: fãs de DRAMA

gosto_filmes = np.random.rand(2, n_filmes)       # quanto cada filme tem de cada gosto
gosto_filmes[1, :4] *= 0.15                      # filmes 0-3: são de AÇÃO
gosto_filmes[0, 4:] *= 0.15                      # filmes 4-7: são de DRAMA

R_grande = gosto_usuarios @ gosto_filmes               # notas "verdadeiras"
R_grande = 5 * R_grande / R_grande.max()               # coloca na escala 0 a 5
R_grande += np.random.normal(0, 0.3, R_grande.shape)   # + ruído do mundo real
R_grande = np.clip(R_grande, 0, 5)

print("Notas de 3 fãs de ação (usuários 0-2):")
print(np.round(R_grande[:3], 1))
print("\nNotas de 3 fãs de drama (usuários 10-12):")
print(np.round(R_grande[10:13], 1))
print("\n(colunas 0-3 = filmes de ação | colunas 4-7 = filmes de drama)")

plt.figure(figsize=(6, 6))
plt.imshow(R_grande, cmap="YlOrRd", aspect="auto")
plt.colorbar(label="nota")
plt.axvline(3.5, color="black", lw=1.5); plt.axhline(9.5, color="black", lw=1.5)
plt.xlabel("filme"); plt.ylabel("usuário")
plt.title("Matriz de notas: consegue ver os dois blocos?")
plt.show()

In [ ]:
# A SVD em 1 linha:
U, S, Vt = np.linalg.svd(R_grande, full_matrices=False)

energia = S**2 / np.sum(S**2)          # quanto cada componente "explica" da matriz

print("Valores singulares:", np.round(S, 2))
print("Energia explicada (%):", np.round(100 * energia, 1))
print("Acumulada (%):       ", np.round(100 * np.cumsum(energia), 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, len(S) + 1), S)
ax1.set_xlabel("Componente"); ax1.set_ylabel("Valor singular")
ax1.set_title("Importância de cada 'gosto latente'")

ax2.plot(range(1, len(S) + 1), 100 * np.cumsum(energia), 'o-')
ax2.axhline(95, color='red', ls='--', lw=1, label="95%")
ax2.set_xlabel("k (componentes usados)"); ax2.set_ylabel("Energia acumulada (%)")
ax2.set_title("Quanto da matriz os k primeiros explicam"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**❓ Pergunta 3.1** — Olhando o gráfico de barras **e** a energia acumulada, quantos componentes parecem realmente importantes? Quanto por cento da matriz os dois primeiros explicam juntos? Isso bate com a forma como os dados foram construídos (releia o comentário do código que gerou `R_grande`)?

**✍️ Sua resposta:**

*(escreva aqui)*

**Exercício 3.2** — Vamos **comprimir** a matriz usando só os `k` primeiros componentes e medir o erro de reconstrução:

$$R_k = U_k \, \Sigma_k \, V_k^T \qquad \text{erro} = \|R - R_k\|$$

In [ ]:
def reconstruir(U, S, Vt, k):
    """Reconstrói a matriz usando só os k primeiros componentes."""
    # 👉 SEU CÓDIGO AQUI (3 linhas): pegue só a parte de cada matriz e multiplique
    #    dica: as colunas de U -> U[:, :k]
    #          os k maiores valores singulares como matriz diagonal -> np.diag(S[:k])
    #          as linhas de Vt -> Vt[:k, :]
    Uk    = ...
    Sigma = ...
    Vtk   = ...
    return Uk @ Sigma @ Vtk

erros = []
for k in range(1, len(S) + 1):
    Rk = reconstruir(U, S, Vt, k)
    erro = np.linalg.norm(R_grande - Rk)
    erros.append(erro)
    print(f"k={k}: erro de reconstrução = {erro:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(S) + 1), erros, 'o-')
plt.xlabel("k (componentes usados)"); plt.ylabel("Erro de reconstrução")
plt.title("Compressão via SVD: quanto perdemos?")
plt.grid(alpha=0.3)
plt.show()

**❓ Pergunta 3.2** — A partir de qual valor de `k` "quase não vale mais a pena" adicionar componentes? Se você tivesse que guardar essa matriz gastando o mínimo de memória, qual `k` escolheria e por quê?

**✍️ Sua resposta:**

*(escreva aqui)*

---
# Parte 4 — Derivadas e Gradiente Descendente: a máquina que aprende 📉

Como um modelo de IA "aprende"? Ele **minimiza uma função de erro** dando pequenos passos na direção contrária à derivada. Esse é o **gradiente descendente**:

$$x_{novo} = x_{atual} - \alpha \cdot f'(x_{atual})$$

onde $\alpha$ é a **taxa de aprendizado** (tamanho do passo).

Vamos começar com uma função simples: $f(x) = (x - 3)^2$. Sabemos que o mínimo é em $x = 3$ — será que o algoritmo encontra sozinho?

**Antes de programar, no papel:** derive $f(x) = (x-3)^2$. Você deve encontrar $f'(x) = 2(x - 3)$.

In [ ]:
def f(x):
    return (x - 3)**2

def derivada_f(x):
    # 👉 SEU CÓDIGO AQUI: escreva a derivada que você calculou no papel
    return ...

# Gradiente descendente
alpha = 0.1        # taxa de aprendizado
x = -5.0           # ponto de partida (bem longe do mínimo)
caminho = [x]

for passo in range(30):
    x = x - alpha * derivada_f(x)   # a fórmula mágica
    caminho.append(x)

print(f"Depois de 30 passos, x = {x:.4f}  (esperado: 3)")
print("Primeiros 6 pontos do caminho:", np.round(caminho[:6], 3))
print("Últimos 3 pontos do caminho:  ", np.round(caminho[-3:], 3))

# Visualização do caminho
xs = np.linspace(-6, 10, 200)
plt.figure(figsize=(8, 5))
plt.plot(xs, f(xs), label="f(x) = (x-3)²")
plt.plot(caminho, [f(c) for c in caminho], 'ro-', ms=4, label="Caminho do algoritmo")
plt.xlim(-8, 12); plt.ylim(0, 80)   # trava a escala: se o caminho "sair da tela", ele divergiu!
plt.legend(); plt.title(f"Gradiente descendente com alpha={alpha}")
plt.show()

**Exercício 4.1** — Rode a célula acima de novo mudando `alpha` para `0.01`, depois para `0.9` e depois para `1.05`. Observe o gráfico **e os números impressos** em cada caso.

> ⚠️ A escala do gráfico está travada de propósito. Se os pontos vermelhos sumirem da tela, olhe os valores impressos: eles estão crescendo ou diminuindo?

**❓ Pergunta 4.1** — O que acontece com o caminho em cada caso? Por que uma taxa de aprendizado grande demais é perigosa?

**✍️ Sua resposta:**

*(escreva aqui)*

**Exercício 4.2 (fechando o ciclo 🎬)** — Agora usamos gradiente descendente para **treinar o recomendador**: encontrar a reta $y = w \cdot x$ que melhor prevê a nota de um usuário para um filme a partir de uma característica do filme. O erro é o **erro quadrático médio**:

$$L(w) = \frac{1}{n} \sum_i (w \, x_i - y_i)^2 \qquad \Rightarrow \qquad L'(w) = \frac{2}{n} \sum_i (w \, x_i - y_i) \, x_i$$

O código está pronto — você só completa a linha do passo do gradiente (a mesma fórmula da Parte 4!).

In [ ]:
# Dados: x = "quantidade de ação" do filme (0 a 1), y = nota que o usuário deu
x_dados = np.random.rand(30)
w_verdadeiro = 4.2
y_dados = w_verdadeiro * x_dados + np.random.normal(0, 0.3, 30)

def gradiente_L(w):
    return (2 / len(x_dados)) * np.sum((w * x_dados - y_dados) * x_dados)

w = 0.0
alpha = 0.5
historico_w = [w]
for passo in range(50):
    # 👉 SEU CÓDIGO AQUI: atualize w com o passo de gradiente descendente
    w = ...
    historico_w.append(w)

print(f"w aprendido = {w:.3f}   |   w verdadeiro = {w_verdadeiro}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(historico_w); a1.set_title("Convergência de w"); a1.set_xlabel("passo"); a1.grid(alpha=0.3)
a2.scatter(x_dados, y_dados, alpha=0.6, label="dados")
xs = np.linspace(0, 1, 10)
a2.plot(xs, w * xs, 'r-', label=f"reta aprendida (w={w:.2f})")
a2.legend(); a2.set_title("Modelo treinado")
plt.show()

---
# Parte 5 (BÔNUS) — Métodos numéricos: e se eu não souber derivar? 🧮

Na prática, nem sempre temos a derivada "no papel". O computador pode **aproximá-la** com **diferenças finitas**:

$$f'(x) \approx \frac{f(x + h) - f(x)}{h} \qquad \text{para } h \text{ pequeno}$$

In [ ]:
def derivada_numerica(f, x, h):
    # 👉 SEU CÓDIGO AQUI: implemente a fórmula de diferenças finitas
    return ...

x0 = 5.0
exata = derivada_f(x0)   # sua derivada analítica da Parte 4

print(f"Derivada exata em x=5: {exata}")
print(f"{'h':>12} | {'aproximação':>12} | {'erro':>12}")
for h in [1, 0.1, 0.01, 1e-4, 1e-8, 1e-12]:
    aprox = derivada_numerica(f, x0, h)
    print(f"{h:>12.0e} | {aprox:>12.6f} | {abs(aprox - exata):>12.2e}")

**❓ Pergunta 5.1** — O erro sempre diminui quando `h` fica menor? O que acontece com `h = 1e-12` e por que isso é um alerta importante sobre computação numérica? *(dica: pesquise "erro de arredondamento em ponto flutuante")*

**✍️ Sua resposta:**

*(escreva aqui)*

---
# Parte 6 — Integrais: somando o erro do modelo 📐

Derivada mede **inclinação**; integral mede **área acumulada**. No mundo da IA, integrais aparecem quando queremos o **erro total** de um modelo ao longo de um intervalo, ou a probabilidade acumulada de uma distribuição (isso volta no Trabalho 2).

Assim como com a derivada, nem sempre dá para integrar "no papel". O computador aproxima a área por **trapézios**: divide o intervalo em `n` pedaços e soma a área de cada trapézio.

$$\int_a^b f(x)\,dx \;\approx\; h\left[\frac{f(x_0)}{2} + f(x_1) + f(x_2) + \dots + f(x_{n-1}) + \frac{f(x_n)}{2}\right], \qquad h = \frac{b-a}{n}$$

**No papel primeiro:** calcule $\int_0^3 (x-3)^2\,dx$ analiticamente. *(dica: a primitiva é $\frac{(x-3)^3}{3}$)* — você deve encontrar **9**.

In [ ]:
def integral_trapezio(f, a, b, n):
    """Aproxima a integral de f entre a e b usando n trapézios."""
    xs = np.linspace(a, b, n + 1)
    h  = (b - a) / n
    # 👉 SEU CÓDIGO AQUI (2 linhas): separe as "pontas" do "meio"
    soma_pontas = ...        # f no primeiro ponto + f no último, divididos por 2
    soma_meio   = ...        # soma de f em todos os pontos do meio -> np.sum(f(xs[1:-1]))
    return h * (soma_pontas + soma_meio)

exata_integral = 9.0   # o valor que você calculou no papel

print(f"{'n':>8} | {'aproximação':>12} | {'erro':>12}")
for n in [1, 2, 4, 8, 16, 64, 256, 1024]:
    aprox = integral_trapezio(f, 0, 3, n)
    print(f"{n:>8} | {aprox:>12.6f} | {abs(aprox - exata_integral):>12.2e}")

# Visualização: os trapézios com n=6
n = 6
xs_t = np.linspace(0, 3, n + 1)
xs_f = np.linspace(0, 3, 200)
plt.figure(figsize=(7, 4))
plt.plot(xs_f, f(xs_f), 'b-', lw=2, label="f(x) = (x-3)²")
plt.fill_between(xs_t, f(xs_t), alpha=0.3, step=None, color='orange', label=f"{n} trapézios")
plt.plot(xs_t, f(xs_t), 'o-', color='darkorange')
plt.legend(); plt.title("Integral por trapézios"); plt.grid(alpha=0.3)
plt.show()

**❓ Pergunta 6.1** — Quando você dobra `n`, o que acontece com o erro (ele cai pela metade? por quatro?).

**✍️ Sua resposta:**

*(escreva aqui)*

**❓ Pergunta 6.2 (conectando com a Parte 5)** — Na derivada numérica, diminuir `h` demais **piorou** o resultado. Aqui, aumentar `n` (ou seja, diminuir `h`) parece só melhorar. Por que os dois métodos se comportam de forma diferente? *(dica: pense em quantas subtrações de números quase iguais cada fórmula faz)*

**✍️ Sua resposta:**

*(escreva aqui)*

---
# ✅ Checklist final antes de entregar

- [ ] Nome e matrícula preenchidos na primeira célula
- [ ] `Kernel > Restart & Run All` executado sem erros
- [ ] Todos os `👉 SEU CÓDIGO AQUI` completados
- [ ] Todas as perguntas ❓ respondidas com suas palavras
- [ ] As derivações "no papel" (derivada da Parte 4, integral da Parte 6) aparecem em alguma célula de texto

**Critérios de avaliação:** código funcionando (40%) • interpretação nas respostas ✍️ (50%) • organização e clareza (10%). As perguntas 1.2 e 6.2 valem até +1,0 ponto de bônus.